In [ ]:
import numpy as np
import pickle as pkl
import os

cot = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/FewShotCOTCLUTRR_Thu_Jan__2_00.41.44_2025_iter2'

cot = np.load(open(cot, 'rb'), allow_pickle=True)

In [ ]:
len(cot)

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
folio_og = load_dataset("yale-nlp/FOLIO")
folio_og = folio_og['train']

In [ ]:
folio_syn = json.load(open('//home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/folio_proofd5_test.json', 'r'))

In [ ]:
folio_new = {}
for f in folio_og:
    folio_new[f['example_id']] = f

In [ ]:
folio_new_write = []
for name in names:
    folio_new_write.append(folio_new[folio_syn[name]['example_id']])
    folio_new_write[-1]['context'] = folio_new_write[-1]['premises'].split('\n')
    folio_new_write[-1]['question'] = folio_new_write[-1]['conclusion']

In [ ]:
json.dump(folio_new_write, open('/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/folio_new.json', 'w') )

In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb

c = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_csvs/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/folio_proofd5_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())

task = 'folio'
nameed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
nameed_list = []
labels = {}
for row in cr:
    if row[2] == 'SAT' and row[3] == 'SAT':
        cnf = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs/neg_'+row[1]).readlines()[0].strip('\n')
        num_clause = int(cnf.split(' ')[-1])
       
        if task=='folio':
            bb = get_bb('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs/'+row[1])
            jb = set(bb['pos']).intersection(set(bb['neg']))
            if len(jb) == 0:
                continue
        # if num_clause > 500:
            # continue
        names.append(int(row[1].split('proofd5')[1].split('.cnf')[0]))
        labels[row[1]] = data[int(row[1].split('proofd5')[1].split('.')[0])]['label']

In [ ]:
print(len(labels))


In [ ]:
int(row[1].split('clutrr')[1].split('.cnf')[0])

In [ ]:
i = 0
cot_acc = 0
cot_preds = {}
for key, value in labels.items():
    if cot[i] == value:
        cot_acc += 1
        cot_preds[key] = True
    else:
        cot_preds[key] = False
    i += 1
print(cot_acc)

In [ ]:
few_shot = "Facts:n[Nancy] likes to cut the hair of her daughter [Heidi].\n[Heidi]'s sister [Lorraine] went to beauty school and taught them all how to cut hair expertly. " + \
            "\nHere are some additional facts and rules we\'ve found:\nNancy is the mother of Lorraine\n If Heidi is the sister of Lorraine and Heidi is the daughter of Nancy then Nancy is the mother of Lorraine.\n" + \
            "Question: Is the following statement true: \n\"[Lorraine] is [Nancy]\'s daughter\"\nAnswer: Let\'s think step by step. \n1. We have already found that Nancy is the mother of Lorraine.\n2. If Nancy is the mother of Lorraine, then Lorraine is the daughter of Nancy.\nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts:\n[Dale] and his sister [Nancy] are decorating for a party.\n[Nancy]'s daughter [Louise] thinks the party will be fun.\n" + \
            "Here are some additional facts and rules we\'ve found:\nDale is the uncle of Louise. If Nancy is the sister of Dale and Nancy is the mother of Louise then Dale is the uncle of Louise.\n" + \
            "Question: Is the following statement true: \n\"[Louise] is not [Dales]\'s niece\"\n" + \
            "Answer: Let\'s think step by step. 1. We are given that Dale is the uncle of Louise.\n2.If Dale is the uncle of Louise, then Louise is the niece of Dale.\nTherefore, the answer is No, the statement is not true.\n***\n" + \
            "Facts: \n[Lillian] and her sister [Nancy] are the only children in their family. \n[Lillian]'s biggest accomplishment is raising her son [Douglas]. " + \
            "\nHere are some additional facts and rules we\'ve found:\nLillian is the sister of Nancy. \nIf Nancy is the sister if Lillian then Lillian is the sister of Nancy.\n" + \
            "Question: Is the following statement true: \n\"[Douglas] is [Nancy]\'s nephew\"\nAnswer: Let\'s think step by step. \n1. [Douglas] is [Lillian]\'s son. \n2. [Nancy] is [Lillian]\'s sister. " + \
            "3\n. [Douglas] is [Nancy]\'s nephew. \nTherefore, the answer to the question is Yes, the statement is true. \n***\n" + \
            "Facts: \n[Ashley] liked to go to the park with her granddaughter [Charlotte]. \n[Dale], [Charlotte]'s father, like to take her to the movies instead. " + \
            "\nHere are some additional facts and rules we\'ve found:\nDale is the son of Ashley. If Dale is father of Charlotte and Ashley is the grandmother of Charlotte then Dale is the son of Ashley.\n" + \
            "Question: Is the following statement true: \n\"[Ashley] is not [Dale]\'s mother\"\nAnswer: Let\'s think step by step. \n1. We are given that Dale is the son of Ashley. \n2. If Dale is the son of Ashley, then Ashley is the mother of Dale. " + \
            "\nTherefore, the answer to the question is No, the statement is ot true.\n***\n"

ans = few_shot + 'a;sldkfj;alskdjf***'
print(ans.split('***')[4])

In [ ]:
a = 'grandson_of_james\'_sibling_James_Donald_'
split = a.split('_')
rel_str = ''
for a in split[:-1]:
    rel_str += a + '-'
rel_str = rel_str[:-1]
print(rel_str)

In [ ]:
import copy
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_folio_8B_preds_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/FewShotCOTFOLIO_Wed_Jan__1_06.25.21_2025_iter'
# cot_iter ='/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_folio_8B_preds_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOTFOLIO_Tue_Apr_29_13.38.54_2025_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/LOT_folio_M7B_preds_iter'
# cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/FewShotCOTFOLIO_Thu_Jan__2_00.41.44_2025_iter'
cot_iter = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/preds/mistral_FewShotCOT_folionew_Mon_May_12_11.51.34_2025_iter'
cot_pred = []
cot_pred_list = []
cot_accs = []
for i in range(20):
    cot = np.load(open(cot_iter + str(i), 'rb'), allow_pickle=True)
    
    cot_acc = 0
    cot_preds = {}
    cot_preds_list = []
    j = 0
    for key, value in labels.items():
        # print(value, [i])
        if cot[j] == value:
            cot_acc += 1
            cot_preds[key] = True
            cot_preds_list.append(1)
        else:
            cot_preds[key] = False
            cot_preds_list.append(0)
        j += 1
    print(cot_acc/len(cot))
    cot_accs.append(cot_acc)
    cot_pred.append(copy.deepcopy(cot_preds))
    cot_pred_list.append(copy.deepcopy(cot_preds_list))

In [ ]:
cot

In [ ]:
cot_pred_list

In [ ]:
n_votes = []
sc_pred = {}
for i in range(len(cot_pred_list[0])):
    n_votes.append(0)
    for j in range(len(cot_pred_list)):
    # for j in range(5):
        n_votes[-1] += cot_pred_list[j][i]
sc_acc = 0
for key, value in cot_pred[0].items():
    tmp = 0
    for j in cot_pred:
        tmp+= j[key]
    if tmp >=np.ceil(len(cot_pred_list)/2+0.5): 
        sc_pred[key] = 1
        sc_acc += 1
    
    else: sc_pred[key]=0

In [ ]:
cot = np.load(open(cot_iter + str(9) + '_[0, 1]', 'rb'), allow_pickle=True).item()
idxd = {'true':0, 'false': 1}
n_votes = []
for key, value in cot.items():
    for key, value in labels.items():
        n_votes.append(cot[key][idxd[labels[key]]]-1)
    # n_votes.append(

In [ ]:
len(cot_pred_list)

In [ ]:
print(np.sum(np.where(np.array(n_votes) >= np.ceil(len(cot_pred_list)/2+0.5), 1, 0) ))
# print(np.sum(np.where(np.array(n_votes) >= 3, 1, 0) ))
# print('sc acc:',np.sum(np.where(np.array(n_votes) >= 3, 1, 0) )/len(cot))

print('sc acc:',np.sum(np.where(np.array(n_votes) >= (np.ceil(len(cot_pred_list)/2 + 0.5)), 1, 0) )/len(cot))
print('cot acc:', np.mean(cot_accs)/len(cot))

In [ ]:
bs_sc = []
bs_sc_acc = []
for i in range(len(n_votes)):
    bs_sc.append(resample(n_votes, n_samples=86))
    bs_sc_acc.append(np.sum(np.where(np.array(bs_sc[-1]) >= (np.ceil((20)/2+0.5)), 1, 0) )/len(bs_sc[-1]))


In [ ]:
# outs_str = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/all_outs_cot_met_folio_rulethresh07_thresh05_varname_llama70B_varname_cotThresh08'
# outs_str = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/all_outs_cot_met_folio_thresh05_rulethresh07_sc20_8B.pkl'
outs_str = '/home/XXXX/XXXX//fs_backup_feb13/LLM-project/all_outs_cot_met_folio_L8B_ablate_all'
outs = np.load(open(outs_str, 'rb'), allow_pickle=True).item()
bs_outs_acc = []
outs_pred = {}
outs_acc = 0
num_trues = 0
for key, value in outs.items():
    if len(value[1]['neg']) == 0 and labels[key] == 'false':
        outs_pred[key] = True
        outs_acc += 1
    elif len(value[1]['pos']) == 0 and labels[key] == 'true':
        outs_pred[key] = True
        outs_acc += 1
    else:
        outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
outs_acc /= len(outs_pred.keys())
outs_pred_val = np.array(list(outs_pred.values()))

for i in range(len(outs_pred)):
    bs_outs_acc.append(np.sum(resample(outs_pred_val, n_samples=86))/86)
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(outs_pred.keys()))
print(len(outs))

In [ ]:
from scipy.stats import wilcoxon
print(wilcoxon(np.array(bs_outs_acc) - np.array(bs_sc_acc), alternative='greater'))

In [ ]:
np.mean(np.array(bs_outs_acc) - np.array(bs_sc_acc))

In [ ]:
np.sum(list(outs_pred_val))

In [ ]:
np.std(bs_outs_acc)

In [ ]:
from scipy import stats

confidence_level=0.95
d = bs_outs_acc
ci = stats.t.interval(confidence_level, df=len(d)-1, loc=np.mean(d), scale=np.std(d, ddof=1) / np.sqrt(len(d)))
print(ci)

In [ ]:
np.mean(bs_outs_acc)

In [ ]:
len(cot_pred_list)

In [ ]:
sc_acc

In [ ]:
outs_str = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/all_outs_cot_met_folio_cotthresh1_05thresh_03rulethresh_70B.pkl'
outs = pkl.load(open(outs_str, 'rb'))

In [ ]:
z = 0
for i in range(len(outs)):
    if len(outs[list(outs.keys())[i]][-1]) > 2:
        # print(outs[list(outs.keys())[i]][-1])
        z += 1
# len(


In [ ]:
print(z)

In [ ]:
len(outs)

In [ ]:
outs_str = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/all_outs_cot_met_folio_cotthresh1_05thresh_03rulethresh_70B.pkl'
outs = pkl.load(open(outs_str, 'rb'))

outs_pred = {}
outs_acc = 0
num_trues = 0
for key, value in outs.items():
    if len(value[1]['neg']) == 0 and labels[key] == 'false':
        outs_pred[key] = True
        outs_acc += 1
    elif len(value[1]['pos']) == 0 and labels[key] == 'true':
        outs_pred[key] = True
        outs_acc += 1
    else:
        outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
outs_acc /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]
print(outs_acc)
print(outs_acc*len(outs_pred.keys()))

In [ ]:
# /home/XXXX/XXXX/fs_backup_feb13/all_outs_thresh09_rulethresh04_contexthresh05_dynamicTure.pkl
missed = pkl.load(open('//home/XXXX/XXXX/fs_backup_feb13//missed_list_' + outs_str, 'rb'))
hunh_list = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/hunh_' + outs_str, 'rb'))

In [ ]:
hunh = []
missed_list = []
for miss in missed:
    missed_list.append(miss[0])
for hunh in hunh_list:
    missed_list.append(hunh)

In [ ]:
len(missed_list)

In [ ]:
miss_acc = 0
miss_sc_acc = 0
no_gains = 0
for miss in missed_list:
    if outs_pred[miss] == True:
        miss_acc += 1
    # print(outs_pred[miss])
    if sc_pred[miss] == True:
        miss_sc_acc += 1
    if sc_pred[miss] == False and sc_pred[miss] == False:
        no_gains += 1
    if sc_pred[miss] == True and outs_pred[miss] == True:
        no_gains += 1
print(miss_acc/len(missed_list))
print(miss_sc_acc/len(missed_list))
# print(no_gains/len(missed_list))
print(miss_sc_acc - miss_acc)

In [ ]:
len(missed_list)

In [ ]:
tt = []
tf = []
ft = []
ff = []
for miss in missed_list:
    if sc_pred[miss] == True and outs_pred[miss] == True:
        tt.append(miss)
    elif sc_pred[miss] == True and outs_pred[miss] == False:
        tf.append(miss)
    elif sc_pred[miss] == False and outs_pred[miss] == True:
        ft.append(miss)
    elif sc_pred[miss] == False and outs_pred[miss] == False:
        ff.append(miss)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in labels.keys():
    # if name in missed_list:continue
    
    if sc_pred[name] == True and outs_pred[name] == True:
        tt.append(name)
    elif sc_pred[name] == True and outs_pred[name] == False:
        tf.append(name)
    elif sc_pred[name] == False and outs_pred[name] == True:
        ft.append(name)
    elif sc_pred[name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
(5

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.hist(n_votes)

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in labels.keys():
    if name in missed_list:continue
    
    if cot_pred[0][name] == True and outs_pred[name] == True:
        tt.append(name)
    elif cot_pred[0][name] == True and outs_pred[name] == False:
        tf.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == True:
        ft.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
tt = []
tf = []
ft = []
ff = []
for name in missed_list:
    # if name in missed_list:continue
    
    if cot_pred[0][name] == True and outs_pred[name] == True:
        tt.append(name)
    elif cot_pred[0][name] == True and outs_pred[name] == False:
        tf.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == True:
        ft.append(name)
    elif cot_pred[0][name] == False and outs_pred[name] == False:
        ff.append(name)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
print(tf)

In [ ]:
print(missed_list)

In [ ]:
scores = pkl.load(open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/scores_temp1_thresh075_thresh05_dynFalse_fixed.pkl', 'rb'))

In [ ]:
print(tf)

In [ ]:
i = 1
for line in outs['clutrr60.cnf'][0]:
    print(line)
    if i % 4 == 3:
        # try:
            # print(line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1]))
        print(scores[line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1])])
        # except:
        #     print(line.split('known predicate: ')[1].split('. Known predicates are')[0].replace('___', line.split('\\box{ ')[1]))
            # print(line)
        #     break
    if i%4 == 0 and not str(line).startswith('calls'):
        # continue
        i = 2
        # print('hihi')

    else: i += 1
# print(outs['clutrr125.cnf'])
# print(outs.keys())

In [ ]:
print(list(scores.keys())[0])

In [ ]:
tt = []
tf = []
ft = []
ff = []
for miss in missed_list:
    if cot_pred[0][miss] == True and outs_pred[miss] == True:
        tt.append(miss)
    elif cot_pred[0][miss] == True and outs_pred[miss] == False:
        tf.append(miss)
    elif cot_pred[0][miss] == False and outs_pred[miss] == True:
        ft.append(miss)
    elif cot_pred[0][miss] == False and outs_pred[miss] == False:
        ff.append(miss)
    
print(len(tt), len(tf), len(ft), len(ff))

In [ ]:
import torch
score_list = []
for score in scores.values():
    score_list.append(torch.stack(score))
score = torch.stack(score_list)

In [ ]:
score

In [ ]:
from matplotlib import pyplot as plt

fig1, ax1 = plt.subplots()
ax1.scatter(x=score[:,0], y=score[:,1], s=3)
ax1.set_xlabel('1 - Does the following rule seem contradictory?')
ax1.set_ylabel('Does the following rule seem contextually relevant?')

In [ ]:
outs_acc*60

In [ ]:
sc_acc

In [ ]:
for key, value in outs_pred.items():
    if key not in missed_list and value == False:
        print(key)

In [ ]:
for i in range(outs['clutrr366.cnf']):
    print(outs['clutrr366.cnf'][i])
# print(outs['clutrr366.cnf'])

In [ ]:
n = 7
for i in range(len(missed[n][1])):
    print(missed[n][1][i])
    # print('\n')

In [ ]:
outs_pred = {}
outs_acc = 0
num_trues = 0
for key, value in outs.items():
    if len(value[1]['neg']) == 0 and labels[key] == 'false':
        outs_pred[key] = True
        outs_acc += 1
    elif len(value[1]['pos']) == 0 and labels[key] == 'true':
        outs_pred[key] = True
        outs_acc += 1
    else:
        outs_pred[key] = False
    if labels[key] == 'true':
        num_trues += 1
outs_acc /= len(outs_pred.keys())
# outs['clutrr545.cnf'][1]

In [ ]:
outs_acc

In [ ]:
outs_pred

In [ ]:
labels[key]

In [ ]:
outs.keys()

In [ ]:
len()

In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb


In [ ]:
import shutil

def get_bb(file, del_sols=None):
    bb = {'pos':  [], 'neg': []}
    
    files = ['/'.join(file.split('/')[:-1]) + '/pos_' + file.split('/')[-1], '/'.join(file.split('/')[:-1]) + '/neg_' + file.split('/')[-1] ]
    for i in range(len(files)):
        file = files[i]
        shutil.copy(file, '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
        if not del_sols==None:
            if 'pos' in file:
                if 'neg' in file:
                    print('l. 416 uh oh')
                      
                ds = del_sols['pos']
            elif 'neg' in file:
                ds = del_sols['neg']
            for sol in ds:
                add_clause('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]))
                cf = open(f'/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]), 'a')
                write_str = '\n'
                for lit in sol:
                    write_str += str(-lit) + ' '
                # write_str += '0'
                cf.write(write_str)
                cf.close()
        # print('running cadical')
        os.system("timeout 5000 /home/XXXX/XXXX/fs_backup_feb13/LLM-project/cadiback/cadiback " + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1]) + '> '  + '/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone")
        #   
        bbone= open('/'.join(file.split('/')[:-2]) + '/tempfiles/' + str(file.split('/')[-1])[:-4] + ".bbone", 'r')
        lines = bbone.readlines()
        #   
        for line in lines:
            if line.startswith('b'):
                #   
                lits = line.split(' ')[1:]
                for lit in lits:
                    lit = lit.strip()
                    if lit == '0':
                        continue
                    lit = int(lit)
                    if 'pos' in file:                                
                        if 'neg' in file:
                            print('l. 447 uh oh')
                              
                        bb['pos'].append(lit)
                    elif 'neg' in file:
                            bb['neg'].append(lit)

    return bb

c = '/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs_csvs_debug/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/XXXX/fs_backup_feb13/SAT-LM/data/unfixed_proofd5_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())

task = 'folio'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
missed_list = []
labels = {}
for row in cr:
    if row[2] == 'SAT' and row[3] == 'SAT':
        cnf = open('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs/neg_'+row[1]).readlines()[0].strip('\n')
        num_clause = int(cnf.split(' ')[-1])
       
        if task=='folio':
            bb = get_bb('/home/XXXX/XXXX/fs_backup_feb13/LLM-project/dimacs/'+row[1])
            jb = set(bb['pos']).intersection(set(bb['neg']))
            if len(jb) == 0:
                continue
        # if num_clause > 500:
            # continue
        names.append(int(row[1].split('proofd5')[1].split('.cnf')[0]))
        labels[row[1]] = data[int(row[1].split('proofd5')[1].split('.')[0])]['label']

In [ ]:
len(names)

In [ ]:
bad_data = []
mistr_data = []
noisy_data=[]
c = '/home/XXXX/LLM-project/dimacs_csvs_debug/solver_finished.csv'
import csv
import json
dataset = '/home/XXXX/SAT-LM/data/unfixed_proofd5_test.json'
with open(dataset, 'r') as df:
    data = json.loads(df.read())
# breakpoint()
task = 'folio'
missed=False
c = open(c, 'r')
cr = csv.reader(c)
names = []
all_outs = {}
missed_list = []
labels = {}
for row in cr:
        if row[2] == 'SAT' and row[3] == 'SAT':
            cnf = open('/home/XXXX/LLM-project/dimacs/neg_'+row[1]).readlines()[0].strip('\n')
            num_clause = int(cnf.split(' ')[-1])
            if row[1] in noisy_data or row[1] in mistr_data:
                continue
            if task=='folio':
                bb = get_bb('/home/XXXX/LLM-project/dimacs/'+row[1])
                jb = set(bb['pos']).intersection(set(bb['neg']))
                if len(jb) == 0:
                    continue
            # if num_clause > 500:
                # continue
            if row[1] in bad_data:
                continue
            names.append(row[1])
            labels[row[1]] = data[int(row[1].split('proofd5')[1].split('.')[0])]['label']
    #   

In [ ]:
len(names)

In [ ]:
labels

In [ ]:
labels.keys()

In [ ]:
import json
folio = json.load(open('/home/XXXX/SAT-LM/data/unfixed_clutrr_test.json', 'r'))
folio[48]

In [ ]:
i = 0
cot_acc = 0
cot_preds = {}
for key, value in labels.items():
    if cot[i] == value:
        cot_acc += 1
        cot_preds[key] = True
    else:
        cot_preds[key] = False
    i += 1
print(cot_acc)

In [ ]:
flipped = 0
flipped_names = []
tf = []
ft = []
for name in names:
    if cot_preds['proofd5' + str(name) + '.cnf'] != outs_pred['proofd5' + str(name) + '.cnf']:
        flipped_names.append('proofd5' + str(name) + '.cnf')
        flipped += 1
    if cot_preds['proofd5' + str(name) + '.cnf'] == True and outs_pred['proofd5' + str(name) + '.cnf'] == False:
        tf.append('proofd5' + str(name) + '.cnf')
    if cot_preds['proofd5' + str(name) + '.cnf'] == False and outs_pred['proofd5' + str(name) + '.cnf'] == True:
        ft.append('proofd5' + str(name) + '.cnf')

print(flipped)
print(len(tf))
print(len(ft))

In [ ]:
flipped = 0
flipped_names = []
tf = []
ft = []
for name in missed_list:
    name = name[7:-4]
    if cot_preds['clutrr' + str(name) + '.cnf'] != outs_pred['clutrr' + str(name) + '.cnf']:
        flipped_names.append('clutrr' + str(name) + '.cnf')
        flipped += 1
    if cot_preds['clutrr' + str(name) + '.cnf'] == True and outs_pred['proofd5' + str(name) + '.cnf'] == False:
        tf.append('proofd5' + str(name) + '.cnf')
    if cot_preds['proofd5' + str(name) + '.cnf'] == False and outs_pred['proofd5' + str(name) + '.cnf'] == True:
        ft.append('proofd5' + str(name) + '.cnf')

print(flipped)
print(len(tf))
print(len(ft))

In [ ]:
missed

In [ ]:
print(tf)

In [ ]:
outs['proofd542.cnf']


In [ ]:
name

In [ ]:
ours = pkl.load(open('/home/XXXX/LLM-project/all_outs_temp1_dynFalse.pkl', 'rb'))

In [ ]:
print(list(ours.keys())[1])
ours[list(ours.keys())[1]]


In [ ]:
list(ours.keys())[5]


In [ ]:
cot

In [ ]:
labels[list(ours.keys())[5]]

In [ ]:
ours[list(ours.keys())[5]]
